# 🔬 KLA Hackathon — NAFNet-SR Image Restoration
**AI-Based Restoration of Degraded Semiconductor Inspection Images**

### Features:
- Direct Google Drive Checkpointing: Weights are saved directly to Google Drive!
- Auto-Resume: If Colab disconnects, re-running Cell 6 automatically resumes from the last saved epoch.
- Live Output Streaming: Keeps Colab active without idle timeouts.


In [ ]:
# CELL 1: Check GPU
!nvidia-smi
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# CELL 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted!')

In [ ]:
# CELL 3: Clone repo & install dependencies
import os
REPO_URL = 'https://github.com/norriy0u/kla-image-restoration.git'
REPO_DIR = '/content/kla_restoration'
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}
!pip install -r requirements.txt -q
print('✓ Setup complete!')

In [ ]:
# CELL 4: Detect dataset paths and prepare local storage
import os, glob, shutil

DRIVE_GT   = '/content/drive/MyDrive/kla_data/train/train/GT'
DRIVE_LR   = '/content/drive/MyDrive/kla_data/train/train/NoisyLR'
DRIVE_TEST = '/content/drive/MyDrive/kla_data/Test_NoisyLR/NoisyLR'

GT_DIR     = '/content/kla_data/GT'
LR_DIR     = '/content/kla_data/NoisyLR'
TEST_DIR   = '/content/kla_data/Test_NoisyLR'

os.makedirs('/content/kla_data', exist_ok=True)
os.makedirs('/content/drive/MyDrive/kla_submission/weights', exist_ok=True)

for src_d, dst_d, label in [(DRIVE_GT, GT_DIR, 'GT'), (DRIVE_LR, LR_DIR, 'NoisyLR'), (DRIVE_TEST, TEST_DIR, 'Test')]:
    n_dst = len(glob.glob(f'{dst_d}/*.npy')) if os.path.exists(dst_d) else 0
    if n_dst < 300:
        print(f'Copying {label} to fast local storage...')
        if os.path.exists(dst_d):
            shutil.rmtree(dst_d)
        shutil.copytree(src_d, dst_d)
    n_dst = len(glob.glob(f'{dst_d}/*.npy'))
    print(f'  ✓ {label}: {n_dst} files ready')

gt_n = len(glob.glob(f'{GT_DIR}/*.npy'))
lr_n = len(glob.glob(f'{LR_DIR}/*.npy'))
assert gt_n >= 3000 and lr_n >= 3000, 'Dataset missing!'
print('✓ Dataset fully prepared!')

In [ ]:
# CELL 5: Quick data inspection
import numpy as np, matplotlib.pyplot as plt, glob, os

gt_files = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))
print(f'GT: {len(gt_files)} | LR: {len(lr_files)}')

n = len(gt_files)
indices = [0, n//3, 2*n//3, n-1]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for col, idx in enumerate(indices):
    gt = np.load(gt_files[idx])
    lr = np.load(lr_files[idx])
    axes[0][col].imshow(gt, cmap='gray', vmin=0, vmax=1)
    axes[0][col].set_title(f'GT #{idx} {gt.shape}\n[{gt.min():.2f},{gt.max():.2f}]', fontsize=9)
    axes[0][col].axis('off')
    axes[1][col].imshow(np.clip(lr,0,1), cmap='gray', vmin=0, vmax=1)
    axes[1][col].set_title(f'NoisyLR #{idx} {lr.shape}\nmax={lr.max():.2f}', fontsize=9)
    axes[1][col].axis('off')

plt.suptitle('Training Pairs — GT (256×256) vs NoisyLR (128×128)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/sample_pairs.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: /content/sample_pairs.png')

In [ ]:
# CELL 6: TRAIN — Direct Drive Checkpointing + Real-time Streaming
# Weights are saved directly to Google Drive: MyDrive/kla_submission/weights
# If disconnect occurs, re-running this cell automatically resumes from last checkpoint!

DRIVE_WEIGHTS = '/content/drive/MyDrive/kla_submission/weights'
os.makedirs(DRIVE_WEIGHTS, exist_ok=True)
%cd /content/kla_restoration

!PYTHONUNBUFFERED=1 python train.py \
    --gt_dir {GT_DIR} \
    --lr_dir {LR_DIR} \
    --epochs 200 \
    --batch_size 8 \
    --num_workers 2 \
    --val_fraction 0.1 \
    --patch_size_gt 256 \
    --model_variant base \
    --weights_dir {DRIVE_WEIGHTS} \
    --log_dir /content/drive/MyDrive/kla_submission/logs \
    --no_amp

In [ ]:
# CELL 7: TensorBoard
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/kla_submission/logs

In [ ]:
# CELL 8: Evaluate on validation set
import os, sys, json, glob
sys.path.insert(0, '/content/kla_restoration')
os.makedirs('/content/val_outputs', exist_ok=True)
DRIVE_WEIGHTS = '/content/drive/MyDrive/kla_submission/weights'
best_pt = os.path.join(DRIVE_WEIGHTS, 'best_model.pt')
weights_path = best_pt if os.path.exists(best_pt) else sorted(glob.glob(os.path.join(DRIVE_WEIGHTS,'checkpoint_epoch*.pt')))[-1]

!python /content/kla_restoration/evaluate.py \
    --input_dir {LR_DIR} \
    --output_dir /content/val_outputs \
    --gt_dir {GT_DIR} \
    --weights {weights_path} \
    --batch_size 8

with open('/content/val_outputs/metrics.json') as f:
    m = json.load(f)
print('\n=== VALIDATION METRICS ===')
for k, v in m.items():
    print(f'  {k}: {v}')

In [ ]:
# CELL 9: Run inference on official test set
import os, json, glob
os.makedirs('/content/test_outputs', exist_ok=True)
DRIVE_WEIGHTS = '/content/drive/MyDrive/kla_submission/weights'
best_pt = os.path.join(DRIVE_WEIGHTS, 'best_model.pt')
weights_path = best_pt if os.path.exists(best_pt) else sorted(glob.glob(os.path.join(DRIVE_WEIGHTS,'checkpoint_epoch*.pt')))[-1]

!python /content/kla_restoration/evaluate.py \
    --input_dir {TEST_DIR} \
    --output_dir /content/test_outputs \
    --weights {weights_path} \
    --batch_size 8

with open('/content/test_outputs/metrics.json') as f:
    print(json.dumps(json.load(f), indent=2))

In [ ]:
# CELL 10: Before → After → GT comparison (for PPT Slide 6)
import numpy as np, matplotlib.pyplot as plt, glob, os

gt_files = sorted(glob.glob(os.path.join(GT_DIR, '*.npy')))
lr_files = sorted(glob.glob(os.path.join(LR_DIR, '*.npy')))
pool = lr_files[:min(300, len(lr_files))]
sample_lr = [f for _, f in sorted([(np.load(f).max(), f) for f in pool], reverse=True)[:4]]

fig, axes = plt.subplots(4, 3, figsize=(15, 20))
titles = ['NoisyLR Input (128×128)', 'NAFNet-SR Output (256×256)', 'Ground Truth (256×256)']
colors = ['#e74c3c', '#2ecc71', '#3498db']

for row, lr_path in enumerate(sample_lr):
    stem    = os.path.splitext(os.path.basename(lr_path))[0]
    gt_path = os.path.join(GT_DIR, f'{stem}.npy')
    out_npy = f'/content/val_outputs/{stem}.npy'
    lr_arr  = np.load(lr_path)
    gt_arr  = np.load(gt_path) if os.path.exists(gt_path) else np.zeros((256,256))
    out_arr = np.load(out_npy)  if os.path.exists(out_npy)  else np.zeros((256,256))
    for col, (arr, t, c) in enumerate(zip([lr_arr, out_arr, gt_arr], titles, colors)):
        axes[row][col].imshow(np.clip(arr,0,1), cmap='gray', vmin=0, vmax=1)
        axes[row][col].set_title(f'{t}\n[{arr.min():.2f},{arr.max():.2f}]',
                                  fontsize=10, color=c, fontweight='bold')
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(f'#{stem}', fontsize=9, rotation=90, labelpad=12)

plt.suptitle('NAFNet-SR: Degraded → Restored → Ground Truth', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/before_after_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved /content/before_after_comparison.png  ← use in PPT Slide 6!')

In [ ]:
# CELL 11: Save final outputs to Google Drive
import shutil, os
DRIVE_OUT = '/content/drive/MyDrive/kla_submission'
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copytree('/content/test_outputs', f'{DRIVE_OUT}/test_outputs', dirs_exist_ok=True)
shutil.copy('/content/before_after_comparison.png', f'{DRIVE_OUT}/before_after_comparison.png')
shutil.copy('/content/sample_pairs.png', f'{DRIVE_OUT}/sample_pairs.png')
print(f'✓ All submission artifacts saved to: {DRIVE_OUT}')